### Running the Experiment

This section outlines the experimental pipeline for fine-tuning CodeBERT to classify source code as human-written or machine-generated. The setup combines the SemEval dataset with additional code samples collected from CodeParrot, which are incorporated as label-0 data, and applies techniques such as dataset balancing and layer-wise learning rate decay (LLRD) to improve generalization.

Below are the main steps of the pipeline, including environment setup, data preprocessing, model training, and evaluation.

### 1. Environment Setup

This section installs the essential libraries required for training, data processing, and evaluation.

**Note:** The specific versions listed below have been tested and verified to work correctly in this environment. Using different versions (especially NumPy or Transformers) may cause compatibility issues.

**Libraries:** NumPy, Pandas, Transformers, Datasets, Accelerate, Scikit-learn.

In [ ]:
!pip install -q --upgrade \
  "numpy<2.0" \
  "pandas<2.3" \
  "transformers>=4.40" \
  "datasets>=2.18" \
  "accelerate>=0.28" \
  scikit-learn



### 2. Environment Configuration and Dependencies

Imports the core libraries required for data handling, model training, and evaluation.

Also ensures reproducibility by setting a fixed random seed and automatically selects the available device (GPU if available, otherwise CPU).

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import torch
from torch import nn

from datasets import load_dataset, DatasetDict, concatenate_datasets
from sklearn.metrics import f1_score, accuracy_score

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)

SEED = 42
set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE


### 3. Dataset Loading

Loads the dataset from the Hugging Face Hub using the specified dataset name and configuration.

The dataset is stored for further preprocessing and training.

In [ ]:
DATASET_NAME = "DaniilOr/SemEval-2026-Task13"
SUBSET = "A"

ds = load_dataset(DATASET_NAME, SUBSET)
ds


### 4. Dataset Preprocessing

Prepares the dataset for training by ensuring consistency across all splits (train, validation, test).

First, a unique identifier (`id`) is added to each sample if it is missing.  
Then, only the relevant columns (`id`, `code`, `label`) are retained, removing any unnecessary information.

This results in a clean and standardized dataset for downstream training and evaluation.

In [ ]:
def ensure_id(split):
    if "id" in split.column_names:
        return split
    return split.add_column("id", list(range(len(split))))

ds = DatasetDict({
    "train": ensure_id(ds["train"]),
    "validation": ensure_id(ds["validation"]),
    "test": ensure_id(ds["test"]),
})

def keep_cols(split):
    cols = ["id", "code", "label"]
    cols = [c for c in cols if c in split.column_names]
    return split.select_columns(cols)

ds["train"] = keep_cols(ds["train"])
ds["validation"] = keep_cols(ds["validation"])
ds["test"] = keep_cols(ds["test"]) 

ds["train"].column_names


### 5. Additional Human Data Collection

Streams data from external datasets (CodeParrot and StarCoder) to collect additional code samples without loading everything into memory.

Relevant fields (content, text, code) are extracted and filtered to keep only valid code snippets (length > 40).
Each sample is labeled as human-generated (label = 0), assuming that external code corpora predominantly contain human-written code.

In this experiment, only the CodeParrot dataset was successfully used, as the StarCoder dataset was not accessible (gated on the Hugging Face Hub).

The successfully loaded subsets are then combined into a single dataset.
If a dataset fails to load, it is skipped without interrupting execution.

In [ ]:
from datasets import load_dataset, Dataset, concatenate_datasets
import itertools

HUMAN_EACH = 80000 
extra_parts = []

def stream_take(ds_stream, n, keys=("content","text","code")):
    rows = []
    for ex in itertools.islice(ds_stream, n):
        code = None
        for k in keys:
            if k in ex and ex[k]:
                code = ex[k]; break
        if code and len(code) > 40:
            rows.append({"code": code, "label": 0})
    return Dataset.from_list(rows)


try:
    s = load_dataset("codeparrot/codeparrot-clean", split="train", streaming=True)
    d = stream_take(s, HUMAN_EACH, keys=("content","text"))
    d = d.add_column("id", list(range(len(d))))
    extra_parts.append(d)
    print("Loaded codeparrot-clean:", len(d))
except Exception as e:
    print("codeparrot-clean failed:", e)


try:
    s = load_dataset("bigcode/starcoderdata", split="train", streaming=True)
    d = stream_take(s, HUMAN_EACH, keys=("content","text","code"))
    d = d.add_column("id", list(range(len(d))))
    extra_parts.append(d)
    print("Loaded starcoderdata:", len(d))
except Exception as e:
    print("starcoderdata skipped/failed:", e)

extra_human = concatenate_datasets(extra_parts) if extra_parts else None
print("extra_human total:", None if extra_human is None else len(extra_human))
print("extra_machine: None (folosim machine doar din competiție)")




### 6. Training Data Preparation and Balancing

Combines the base training dataset with additional human samples (if available).

The merged dataset is split into human (`label = 0`) and machine-generated (`label = 1`) samples.  
Both subsets are shuffled and truncated to the same size, defined by the smaller class.

Finally, the balanced subsets are merged and shuffled to create the final training dataset.

In [ ]:
train_base = ds["train"]

parts = [train_base]
if extra_human is not None:
    parts.append(extra_human)

train_mix = concatenate_datasets(parts)

human = train_mix.filter(lambda x: x["label"] == 0)
machine = train_mix.filter(lambda x: x["label"] == 1)

n = min(len(human), len(machine))

human = human.shuffle(seed=SEED).select(range(n))
machine = machine.shuffle(seed=SEED).select(range(n))

train_bal = concatenate_datasets([human, machine]).shuffle(seed=SEED)

print("Train size:", len(train_bal))


### 7. Tokenization and Data Formatting
Initializes the tokenizer from the pretrained microsoft/codebert-base model and defines a maximum sequence length of 256.

The dataset is tokenized by encoding the code field. Truncation is applied to ensure sequences do not exceed the maximum length, and the process is performed in batches for improved efficiency.

Padding is applied dynamically at batch time using DataCollatorWithPadding.

Each dataset split (train, validation, and test) is then converted to PyTorch format, keeping only the necessary columns:

Train & Validation: input_ids, attention_mask, label  
Test: input_ids, attention_mask, label, id

In [ ]:
MODEL_NAME = "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LEN = 256

def tokenize(batch):
    return tokenizer(
        batch["code"],
        truncation=True,
        max_length=MAX_LEN
    )

tokenized = DatasetDict({
    "train": train_bal.map(tokenize, batched=True),
    "validation": ds["validation"].map(tokenize, batched=True),
    "test": ds["test"].map(tokenize, batched=True),
})

tokenized["train"].set_format("torch", columns=["input_ids","attention_mask","label"])
tokenized["validation"].set_format("torch", columns=["input_ids","attention_mask","label"])
tokenized["test"].set_format("torch", columns=["input_ids","attention_mask","label","id"])

data_collator = DataCollatorWithPadding(tokenizer)



### 8. Model (CodeBERT Binary Classifier)

Defines a binary classification model based on the pretrained transformer encoder (microsoft/codebert-base).

The encoder generates contextual representations, from which the representation of the first token from the last hidden state is used as a pooled representation of the input.

A dropout layer (0.1) is applied for regularization, followed by a linear classification head that maps the hidden size to 2 output classes (human vs. machine).

During training, Cross-Entropy Loss is computed when labels are provided.

The model returns a dictionary containing the loss and logits, and is instantiated and moved to the selected device (GPU or CPU).

In [ ]:
class CodeBERTBinary(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden, 2)

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:,0]
        logits = self.classifier(self.dropout(pooled))

        loss=None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return {"loss":loss, "logits":logits}

model = CodeBERTBinary().to(DEVICE)


### 9. Evaluation Metrics

Defines a custom evaluation function used during model validation and testing.

- **Predictions Extraction**: Converts model output logits into predicted class labels using `argmax` over the class dimension.

- **Macro F1 Score**: Computes the macro-averaged F1 score, which gives equal importance to each class regardless of class imbalance.

- **Accuracy**: Measures the overall proportion of correctly predicted labels.

The function returns a dictionary containing:
- `macro_f1`: Macro-averaged F1 score
- `accuracy`: Classification accuracy

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "macro_f1": f1_score(labels, preds, average="macro"),
        "accuracy": accuracy_score(labels, preds),
    }


### 10. Custom Trainer with Layer-wise Learning Rate Decay (LLRD)

Defines a custom `Trainer` that implements Layer-wise Learning Rate Decay (LLRD) applied to the encoder layers, with a separate learning rate for the classification head.

- **Motivation**: Lower layers of the transformer capture general language features, while higher layers and the classification head are more task-specific. LLRD assigns different learning rates to different layers accordingly.

- **Classification Head**: Uses a higher learning rate (`2e-4`) to allow faster adaptation to the task.

- **Encoder Layers**: Each transformer layer is assigned a progressively smaller learning rate using an exponential decay factor (`0.85`), starting from a base learning rate (`2e-5`).

- **Layer-wise Decay Strategy**:  
  The learning rate for layer *i* is computed as:  
  `lr = base_lr * (decay ** (n_layers - 1 - i))`  
  This ensures deeper layers (closer to output) learn faster than earlier layers.

- **Weight Decay**: A weight decay of `0.01` is applied to the optimizer parameter groups.

- **Optimizer**: Uses the `AdamW` optimizer with custom parameter groups.

This approach improves stability and performance when fine-tuning pretrained transformer models.

Note: The embedding layer is excluded from the optimizer parameter groups, meaning it remains frozen during training, while only the transformer layers and classification head are fine-tuned.

In [ ]:
from torch.optim import AdamW

class LLRDTrainer(Trainer):
    def create_optimizer(self):
        if self.optimizer is not None:
            return self.optimizer

        layers = self.model.encoder.encoder.layer
        n_layers = len(layers)

        base_lr = 2e-5
        head_lr = 2e-4
        decay = 0.85

        groups = []

        groups.append({
            "params": self.model.classifier.parameters(),
            "lr": head_lr,
            "weight_decay":0.01
        })
        for i, layer in enumerate(layers):
            lr = base_lr * (decay ** (n_layers-1-i))
            groups.append({
                "params": layer.parameters(),
                "lr": lr,
                "weight_decay":0.01
            })

        self.optimizer = AdamW(groups)
        return self.optimizer


### 11. Training Configuration and Execution

This section orchestrates the fine-tuning process by integrating the custom `LLRDTrainer` with a specific set of configurations defined via `TrainingArguments`.

#### Training Strategy

- **Epoch-based Workflow**: Evaluation and checkpoint saving are performed at the end of each epoch, ensuring consistent monitoring of model performance.

- **Model Selection**: The trainer tracks the `macro_f1` metric and automatically reloads the best-performing model at the end of training (`load_best_model_at_end`).

- **Checkpoint Management**: A limit of 3 checkpoints is maintained to balance storage efficiency while preserving the ability to revert to previous states if needed.

#### Optimization and Hyperparameters

- **Device-dependent batching**: Uses batch sizes of 16 for GPU and 4 for CPU.

- **Warmup Strategy**: Uses a warmup ratio of 0.06, handled internally by the Hugging Face Trainer

#### Execution

The model is trained for 3 epochs on the tokenized datasets. The training process is initiated via the `trainer.train()` method.

In [ ]:
args = TrainingArguments(
    output_dir="./codebert_ckpt",

    eval_strategy="epoch", 
    save_strategy="epoch",
    save_total_limit=3,

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    num_train_epochs=3,

    per_device_train_batch_size=16 if DEVICE=="cuda" else 4,
    per_device_eval_batch_size=32 if DEVICE=="cuda" else 4,

    warmup_ratio=0.06,
    fp16=(DEVICE=="cuda"),
    report_to="none",
)

trainer = LLRDTrainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()



### 12. Validation Evaluation

Evaluates the trained model on the validation dataset using the trainer’s `evaluate` method.

- Computes evaluation metrics, including `macro_f1`.
- Extracts and prints the validation Macro F1 score for performance assessment.

In [ ]:
val_metrics = trainer.evaluate(tokenized["validation"])
print("VALIDATION macro-F1:", val_metrics["eval_macro_f1"])

### 13. Validation Output

Evaluates the trained model on the validation dataset using the trainer’s `evaluate` method.

- Computes evaluation metrics, including `macro_f1`.
- Prints the validation Macro F1 score.

The output should resemble a high Macro F1 score (e.g., around 0.9+), along with the evaluation progress, indicating strong model performance on the validation set.

### 14. Test Evaluation

Evaluates the trained model on the test dataset using the trainer’s `evaluate` method.

- Computes evaluation metrics, including `macro_f1`.
- Extracts and prints the test Macro F1 score for performance assessment.

In [ ]:
test_metrics = trainer.evaluate(tokenized["test"])
print("TEST macro-F1:", test_metrics["eval_macro_f1"])


### 15. Test Output

The output should indicate a lower Macro F1 score (e.g., around 0.44), reflecting the model’s performance on unseen test data.